# Détection d'anomalies routières — Étape 5 : Déploiement

**Notebook :** `05_deploiement.ipynb`

Le benchmark (étape 4) a retenu **C. DeepLabV3+ pré-entraîné** : meilleure AP (0,830), plus
stable d'un tour à l'autre, meilleure généralisation aux pneus jamais vus, et le plus léger des
deux modèles supervisés. Cette étape le transforme en **application utilisable** :

| § | Étape | Résultat |
|---|---|---|
| 0 | Chargement | données + `modele_retenu.json` de l'étape 4 |
| 1 | Entraînement final sur les 60 images | le modèle de production |
| 2 | Export : PyTorch et **ONNX** | un fichier de modèle portable, sans PyTorch |
| 3 | Vérification de l'export et mesure de vitesse | ONNX = PyTorch ; temps d'inférence |
| 4 | **Application web Gradio** dans Colab | un lien public de démonstration (72 h) |
| 5 | Paquet de l'application | l'application qui tourne sur votre PC, sans Colab |

**GPU recommandé** pour l'entraînement final (≈ 3 minutes sur T4).

## 0. Chargement

Sélectionnez **en une seule fois** : `RoadAnomaly.zip`, `roadanomaly_lib.py` et
`resultats_etape4.zip` (qui contient `modele_retenu.json`).

In [ ]:
import os, sys, zipfile, shutil, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
BASE = Path("/content") if IN_COLAB else Path.cwd()
OUT = BASE / "resultats_etape5"; OUT.mkdir(exist_ok=True)

def trouver_dataset(dossier):
    dossier = Path(dossier)
    hits = sorted(dossier.rglob("frame_list.json")) if dossier.exists() else []
    return hits[0].parent if hits else None

if IN_COLAB:
    os.chdir(BASE)
    if trouver_dataset(BASE / "data") is None or not (BASE / "etape4" / "modele_retenu.json").exists():
        from google.colab import files
        print("Sélectionnez RoadAnomaly.zip, roadanomaly_lib.py et resultats_etape4.zip :")
        for nom in files.upload():
            p = BASE / nom
            if nom.startswith("RoadAnomaly") and nom.endswith(".zip"):
                with zipfile.ZipFile(p) as z: z.extractall(BASE / "data")
                os.remove(p)
            elif nom.startswith("resultats_etape4") and nom.endswith(".zip"):
                with zipfile.ZipFile(p) as z: z.extractall(BASE / "etape4")
                os.remove(p)
            elif nom.startswith("roadanomaly_lib") and nom.endswith(".py") and nom != "roadanomaly_lib.py":
                shutil.move(p, BASE / "roadanomaly_lib.py")
    DATA_ROOT = trouver_dataset(BASE / "data")
else:
    DATA_ROOT = next((p for p in [Path("RoadAnomaly"), Path("../RoadAnomaly")]
                      if (p / "frame_list.json").exists()), None)

assert DATA_ROOT is not None, "Dataset introuvable : envoyez RoadAnomaly.zip"
assert (BASE / "roadanomaly_lib.py").exists(), "roadanomaly_lib.py manquant"
assert (BASE / "etape4" / "modele_retenu.json").exists(), "modele_retenu.json manquant : envoyez resultats_etape4.zip"
BEST = json.load(open(BASE / "etape4" / "modele_retenu.json", encoding="utf-8"))
print(json.dumps(BEST, indent=1, ensure_ascii=False))

In [ ]:
import importlib
def ensure(module, pip_name):
    try:
        importlib.import_module(module)
    except ImportError:
        %pip install -q {pip_name}
        importlib.invalidate_caches()
ensure("segmentation_models_pytorch", "segmentation-models-pytorch==0.6.0")
ensure("onnxruntime", "onnxruntime")
ensure("onnx", "onnx")
ensure("gradio", "gradio")
try:
    import albumentations
    assert int(albumentations.__version__.split(".")[0]) >= 2
except (ImportError, AssertionError):
    %pip install -q "albumentations==2.0.8"
    importlib.invalidate_caches()

import time, numpy as np, pandas as pd, torch, torch.nn as nn, cv2, matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
sys.path.insert(0, str(BASE))
import roadanomaly_lib; importlib.reload(roadanomaly_lib)
from roadanomaly_lib import *
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
frames = list_frames(DATA_ROOT)
print("appareil :", device, "| images :", len(frames))
plt.rcParams.update({"figure.dpi": 80, "savefig.dpi": 150, "savefig.bbox": "tight",
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e4e3df", "axes.axisbelow": True})

## 1. Entraînement final sur les 60 images

### Pourquoi réentraîner ?

Pendant la validation croisée, chaque modèle n'a vu que ≈ 36 images : 24 étaient réservées à la
validation et au test. Pour le modèle de production, on utilise **toutes les données** : plus
d'exemples, donc un meilleur modèle. C'est la pratique standard.

### Sans validation, comment choisir l'époque et le seuil ?

Il n'y a plus d'images de côté. On reprend donc les réglages **mesurés pendant la validation
croisée** (`modele_retenu.json`) :

- **nombre d'époques** = la meilleure époque moyenne des 5 tours. On garde le même planning
  (décroissance cosinus prévue sur 40 époques), arrêté à cette époque, pour reproduire les
  conditions qui ont donné les meilleurs modèles ;
- **seuil de décision** = le seuil moyen choisi sur les validations des 5 tours.

### Quelle performance annoncer ?

Le modèle final n'a **plus de jeu de test** : toutes les images ont servi à l'entraîner.
La performance à annoncer est donc **celle de la validation croisée** (AP 0,830, IoU
0,656 ± 0,055), qui estime ce que ce type de modèle obtient sur des images qu'il n'a pas vues.
Évaluer le modèle final sur ses propres images d'entraînement donnerait un score trompeur.

In [ ]:
assert BEST["architecture"] in ("Unet", "DeepLabV3Plus"), \
    "Ce notebook déploie un modèle supervisé (A, B ou C) ; adaptez-le pour un autre choix."
FINAL_EPOCHS = max(1, int(round(BEST["meilleure_epoque_moyenne"])))
THRESHOLD = float(BEST["seuil_moyen"])
PRETRAINED = BEST["key"] != "A_unet_scratch"
ENCODER_WEIGHTS = None if (os.environ.get("RA_NO_PRETRAINED") or not PRETRAINED) else "imagenet"

class FinalConfig:
    epochs_schedule = 40                  # même planning cosinus qu'en validation croisée
    epochs = FINAL_EPOCHS                 # arrêt à la meilleure époque moyenne
    freeze_epochs = 3
    batch_size = 4
    samples_per_image = 4
    lr_decoder = 1e-3
    lr_encoder = 1e-4 if PRETRAINED else 1e-3

print(f"Modèle : {BEST['label']} | {FinalConfig.epochs} époques | seuil {THRESHOLD:.3f}")

In [ ]:
def build_model():
    Arch = getattr(smp, BEST["architecture"])
    return Arch(encoder_name=BEST["encodeur"], encoder_weights=ENCODER_WEIGHTS, in_channels=3, classes=1)

dice_loss = smp.losses.DiceLoss(mode="binary", from_logits=True)
bce_loss = nn.BCEWithLogitsLoss()
criterion = lambda logits, y: bce_loss(logits, y) + dice_loss(logits, y)

import random
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
cfg = FinalConfig
ds = RoadAnomalyDataset(DATA_ROOT, frames, get_train_transform(), cfg.samples_per_image)
loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True, num_workers=2, drop_last=True)
model = build_model().to(device)
enc = list(model.encoder.parameters())
dec = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
opt = torch.optim.AdamW([{"params": enc, "lr": cfg.lr_encoder}, {"params": dec, "lr": cfg.lr_decoder}],
                        weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs_schedule)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

hist, t0 = [], time.time()
for epoch in range(1, cfg.epochs + 1):
    frozen = PRETRAINED and epoch <= cfg.freeze_epochs
    for p in enc:
        p.requires_grad = not frozen
    model.train()
    if frozen:
        model.encoder.eval()
    tot, n = 0.0, 0
    for b in loader:
        x, y = b["image"].to(device), b["mask"].to(device)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item() * len(x); n += len(x)
    sched.step()
    hist.append(tot / n)
    print(f"époque {epoch:2d}/{cfg.epochs}  perte {tot / n:.3f}{'  (encodeur gelé)' if frozen else ''}")
print(f"Entraînement final : {(time.time() - t0) / 60:.1f} min sur {len(frames)} images")
model.eval()

fig, ax = plt.subplots(figsize=(6.5, 3))
ax.plot(range(1, len(hist) + 1), hist, color="#2a78d6", lw=2)
ax.set_xlabel("Époque"); ax.set_ylabel("Perte (BCE + Dice)"); ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.set_title("Entraînement final sur les 60 images", fontweight="bold")
plt.tight_layout(); fig.savefig(OUT / "courbe_entrainement_final.png"); plt.show()

## 2. Export du modèle

Deux formats :

| Fichier | Format | Usage |
|---|---|---|
| `model.pt` | poids PyTorch (`state_dict`) | réentraîner ou modifier le modèle plus tard |
| `model.onnx` | **ONNX** (*Open Neural Network Exchange*) | le **déploiement** : le modèle s'exécute avec `onnxruntime`, une bibliothèque légère, **sans PyTorch**, sur CPU ou GPU, et même depuis d'autres langages (C++, C#, JavaScript…) |

L'export ONNX déclare la hauteur et la largeur de l'image comme **dimensions variables** :
l'application accepte des images de toute taille.

`config.json` rassemble tout ce dont l'application a besoin : architecture, seuil, normalisation,
résolution de travail et performances de validation croisée (affichées dans l'application).

In [ ]:
APP_DIR = OUT / "app_roadanomaly"; APP_DIR.mkdir(exist_ok=True)
torch.save(model.state_dict(), OUT / "model.pt")

model_cpu = build_model().eval()
model_cpu.load_state_dict({k: v.cpu() for k, v in model.state_dict().items()})
dummy = torch.zeros(1, 3, 544, 960)
export_args = dict(input_names=["image"], output_names=["logits"], opset_version=17,
                   dynamic_axes={"image": {2: "hauteur", 3: "largeur"}, "logits": {2: "hauteur", 3: "largeur"}})
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    try:
        torch.onnx.export(model_cpu, dummy, str(APP_DIR / "model.onnx"), dynamo=False, **export_args)
    except TypeError:                                # anciennes versions de PyTorch : pas d'argument dynamo
        torch.onnx.export(model_cpu, dummy, str(APP_DIR / "model.onnx"), **export_args)

config = {"label": BEST["label"], "architecture": BEST["architecture"], "encoder": BEST["encodeur"],
          "threshold": THRESHOLD, "work_width": WORK_W, "mean": list(MEAN), "std": list(STD),
          "cv_AP": BEST["AP"], "cv_IoU_mean": BEST["IoU_tours_moyenne"], "cv_IoU_std": BEST["IoU_tours_ecart_type"],
          "final_epochs": FINAL_EPOCHS, "trained_on_images": len(frames),
          "dataset": "RoadAnomaly (Lis et al., ICCV 2019)"}
json.dump(config, open(APP_DIR / "config.json", "w", encoding="utf-8"), indent=1, ensure_ascii=False)
for f in ["model.onnx", "config.json"]:
    print(f"{f:<12} {(APP_DIR / f).stat().st_size / 1e6:6.1f} Mo")
print(f"{'model.pt':<12} {(OUT / 'model.pt').stat().st_size / 1e6:6.1f} Mo")

## 3. Vérification de l'export et vitesse

Un export peut silencieusement changer les résultats. On compare donc les sorties ONNX et
PyTorch sur de vraies images, à deux tailles, puis on mesure le temps d'inférence.

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession(str(APP_DIR / "model.onnx"), providers=["CPUExecutionProvider"])
norm = lambda im: ((im.astype(np.float32) / 255 - np.array(MEAN, np.float32)) / np.array(STD, np.float32)).transpose(2, 0, 1)[None]

rows = []
for n in frames[:3]:
    img = load_image(DATA_ROOT, n)
    for (W_, H_) in [(960, 544), (1280, 736)]:
        x = norm(cv2.resize(img, (W_, H_), interpolation=cv2.INTER_AREA))
        o = sess.run(None, {"image": x})[0]
        with torch.no_grad():
            t = model_cpu(torch.from_numpy(x)).numpy()
        rows.append({"image": n[:25], "taille": f"{W_}x{H_}", "écart max (logits)": float(np.abs(o - t).max()),
                     "décisions identiques": float(((o > 0) == (t > 0)).mean())})
check = pd.DataFrame(rows); display(check)
assert check["décisions identiques"].min() > 0.999, "l'export ONNX ne reproduit pas le modèle PyTorch"
print("Export ONNX vérifié : mêmes décisions que PyTorch.")

x = norm(cv2.resize(load_image(DATA_ROOT, frames[0]), (960, 544)))
def chrono(fn, rep=5):
    fn(); t0 = time.time()
    for _ in range(rep): fn()
    return (time.time() - t0) / rep * 1000
vit = {"ONNX, CPU": chrono(lambda: sess.run(None, {"image": x}))}
if device.type == "cuda":
    xg = torch.from_numpy(x).to(device)
    def gpu():
        with torch.no_grad(): model(xg); torch.cuda.synchronize()
    vit["PyTorch, GPU"] = chrono(gpu, 20)
for k, v in vit.items():
    print(f"Temps par image 960x544 — {k:<13}: {v:6.0f} ms  ({1000 / v:5.1f} images/s)")
json.dump(vit, open(OUT / "vitesse_inference.json", "w"), indent=1, ensure_ascii=False)

## 4. Application web (Gradio)

L'application est écrite dans **un seul fichier, `app.py`**, utilisé tel quel ici et en
déploiement permanent (section 5). Elle :

1. reçoit une photo de route de **n'importe quelle taille** ;
2. la ramène à 960 px de large (la résolution d'entraînement), la normalise et la passe au
   modèle ONNX ;
3. remet la probabilité à la taille d'origine, applique le **seuil** (réglable par un curseur) ;
4. affiche les anomalies **en rouge**, chaque zone **encadrée**, la **carte de probabilité**, et
   un résumé : nombre de zones, part de l'image, temps d'analyse.

Elle utilise **`onnxruntime`** et non PyTorch : c'est exactement le code qui tournera en ligne.

In [ ]:
%%writefile {APP_DIR}/app.py
"""Détection d'anomalies routières — application de démonstration.

Modèle : DeepLabV3+ (encodeur ResNet34 pré-entraîné sur ImageNet), entraîné sur le dataset
RoadAnomaly (Lis et al., ICCV 2019), exporté au format ONNX. Fonctionne sans PyTorch.
"""
import json
import time
from pathlib import Path

import cv2
import gradio as gr
import numpy as np
import onnxruntime as ort

HERE = Path(__file__).resolve().parent
CFG = json.load(open(HERE / "config.json", encoding="utf-8"))
SESSION = ort.InferenceSession(str(HERE / "model.onnx"), providers=ort.get_available_providers())
MEAN = np.array(CFG["mean"], np.float32)
STD = np.array(CFG["std"], np.float32)


def predict_proba(img):
    """Image RGB (H, W, 3) uint8, de taille quelconque -> probabilité d'anomalie (H, W)."""
    H, W = img.shape[:2]
    w = CFG["work_width"]
    h = max(32, int(round(H * w / W)))
    x = cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)
    x = np.pad(x, ((0, (-h) % 32), (0, (-w) % 32), (0, 0)))           # multiple de 32, zéros
    x = ((x.astype(np.float32) / 255 - MEAN) / STD).transpose(2, 0, 1)[None]
    logits = SESSION.run(None, {"image": x})[0][0, 0, :h, :w]
    p = 1 / (1 + np.exp(-logits))
    return cv2.resize(p, (W, H), interpolation=cv2.INTER_LINEAR)


def analyse(img, seuil):
    if img is None:
        return None, None, "Ajoutez une image de route."
    img = np.ascontiguousarray(img[..., :3]).astype(np.uint8)
    t0 = time.time()
    p = predict_proba(img)
    ms = (time.time() - t0) * 1000
    mask = (p >= seuil).astype(np.uint8)

    H, W = mask.shape
    k, lab, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    min_area = max(30, 1e-4 * H * W)
    objets = [stats[i] for i in range(1, k) if stats[i, cv2.CC_STAT_AREA] >= min_area]

    vis = img.copy()
    zone = mask.astype(bool)
    vis[zone] = (0.45 * vis[zone] + 0.55 * np.array([255, 60, 40])).astype(np.uint8)
    epais = max(2, W // 400)
    for x, y, w, h, _ in objets:
        cv2.rectangle(vis, (int(x), int(y)), (int(x + w), int(y + h)), (255, 220, 0), epais)
    heat = cv2.applyColorMap((p * 255).astype(np.uint8), cv2.COLORMAP_INFERNO)[..., ::-1]

    if objets:
        texte = (f"**{len(objets)} zone(s) anormale(s) détectée(s)**, soit {mask.mean():.1%} de l'image. "
                 f"Analyse en {ms:.0f} ms.")
    else:
        texte = f"**Aucune anomalie détectée** au seuil {seuil:.2f}. Analyse en {ms:.0f} ms."
    return vis, heat, texte


ENTETE = f"""
# Détection d'anomalies sur la route

Envoyez une photo de route : le modèle signale en **rouge** les objets inattendus (animaux,
rochers, cônes, pneus, obstacles), encadrés en jaune. La carte de droite montre la probabilité
d'anomalie de chaque pixel.

**Modèle :** {CFG["label"]} · validation croisée en 5 plis sur RoadAnomaly :
AP {CFG["cv_AP"]:.3f}, IoU {CFG["cv_IoU_mean"]:.3f} ± {CFG["cv_IoU_std"]:.3f}.

*Projet étudiant de deep learning. Entraîné sur 60 images seulement : les résultats sur des
scènes très différentes peuvent être mauvais. Ce n'est pas un système de sécurité.*
"""

with gr.Blocks(title="Détection d'anomalies routières") as demo:
    gr.Markdown(ENTETE)
    with gr.Row():
        entree = gr.Image(type="numpy", label="Image de route")
        with gr.Column():
            sortie = gr.Image(label="Anomalies détectées")
            carte = gr.Image(label="Probabilité d'anomalie (clair = élevée)")
    seuil = gr.Slider(0.05, 0.95, value=round(CFG["threshold"], 2), step=0.01,
                      label="Seuil de décision (plus bas = plus de détections, plus de fausses alarmes)")
    bouton = gr.Button("Analyser", variant="primary")
    info = gr.Markdown()
    bouton.click(analyse, inputs=[entree, seuil], outputs=[sortie, carte, info])
    seuil.release(analyse, inputs=[entree, seuil], outputs=[sortie, carte, info])

if __name__ == "__main__":
    demo.launch()

In [ ]:
# Test de l'application sans interface, sur une image du dataset
sys.path.insert(0, str(APP_DIR))
import app; importlib.reload(app)
img = load_image(DATA_ROOT, "animals22_red_fox_on_road")
vis, heat, texte = app.analyse(img, app.CFG["threshold"])
print(texte)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
for a, im, t in zip(axes, [img, vis, heat], ["Image", "Anomalies détectées", "Probabilité"]):
    a.imshow(im); a.set_title(t); a.axis("off")
plt.tight_layout(); fig.savefig(OUT / "exemple_application.png"); plt.show()

**Lancer l'application.** La cellule suivante démarre l'interface et affiche un **lien public**
(`https://….gradio.live`), valable 72 heures tant que le notebook tourne. Ouvrez-le dans votre
navigateur ou sur votre téléphone, et testez des **photos que le modèle n'a jamais vues**
(cherchez par exemple « cow on road » ou « rock on highway » sur un moteur de recherche
d'images) : les images du dataset ont servi à l'entraîner, elles donneraient une impression
trop optimiste.

Pour arrêter l'application : interrompez la cellule (bouton carré).

In [ ]:
if IN_COLAB:
    app.demo.launch(share=True, debug=True)
else:
    print("Hors Colab : lancez « python app.py » dans le dossier", APP_DIR)

## 5. Paquet de l'application et exécution sur votre ordinateur

Le lien Colab disparaît quand la session s'arrête. La cellule suivante prépare le dossier
complet **`app_roadanomaly/`**, qui fait tourner l'application **sur n'importe quel ordinateur**,
sans Colab ni GPU :

| Fichier | Rôle |
|---|---|
| `app.py` | l'application |
| `model.onnx` | le modèle |
| `config.json` | seuil, normalisation, performances |
| `requirements.txt` | les bibliothèques à installer |
| `README.md` | description, performances et limites |

**Pas d'images du dataset dans le paquet :** le readme de RoadAnomaly précise que les images sont
fournies **pour la recherche** et appartiennent à leurs auteurs.

### Lancer l'application sur votre PC (gratuit, sans internet)

1. Décompressez `app_roadanomaly.zip` (téléchargé à la fin de ce notebook).
2. Ouvrez un terminal dans ce dossier, puis :
   ```
   pip install gradio onnxruntime opencv-python-headless
   python app.py
   ```
3. Ouvrez **http://127.0.0.1:7860** dans votre navigateur.

C'est la solution la plus sûre pour une **démonstration en soutenance** : elle ne dépend ni
d'internet ni d'une session Colab.

### Hébergement en ligne permanent (facultatif)

Le même dossier peut être déposé tel quel sur **Hugging Face Spaces** (SDK Gradio), le fichier
`README.md` contenant déjà la fiche du Space. Attention : depuis mi-2026, héberger un Space
Gradio sur le processeur gratuit demande un **abonnement PRO** pour les nouveaux comptes. Ce
n'est pas nécessaire pour ce projet.

In [ ]:
import gradio
(APP_DIR / "requirements.txt").write_text("gradio\nonnxruntime\nopencv-python-headless\nnumpy\n", encoding="utf-8")
readme = f'''---
title: Détection d'anomalies routières
emoji: 🚧
colorFrom: blue
colorTo: red
sdk: gradio
sdk_version: {gradio.__version__}
app_file: app.py
pinned: false
---

# Détection d'anomalies routières

Segmentation des objets inattendus sur la route (animaux, rochers, cônes, pneus, obstacles).

- **Modèle :** {config["label"]} (encodeur ResNet34 pré-entraîné ImageNet), exporté en ONNX.
- **Données :** RoadAnomaly (Lis et al., « Detecting the Unexpected via Image Resynthesis », ICCV 2019), 60 images.
- **Performance (validation croisée en 5 plis) :** AP {config["cv_AP"]:.3f}, IoU {config["cv_IoU_mean"]:.3f} ± {config["cv_IoU_std"]:.3f}.
- **Seuil par défaut :** {config["threshold"]:.2f} (réglable dans l'interface).

## Limites

Projet étudiant. Entraîné sur 60 images : les performances peuvent être faibles sur des scènes
éloignées du dataset (nuit, pluie, autres types de caméras). Ne pas utiliser comme système de sécurité.
'''
(APP_DIR / "README.md").write_text(readme, encoding="utf-8")

# test : l'application démarre depuis son dossier, dans un processus séparé (comme sur votre PC)
import subprocess
r = subprocess.run([sys.executable, "-c", "import app; print('app.py OK :', app.CFG['label'])"],
                   cwd=APP_DIR, capture_output=True, text=True)
print(r.stdout or r.stderr[-800:])

shutil.rmtree(APP_DIR / "__pycache__", ignore_errors=True)       # fichiers Python temporaires
shutil.make_archive(str(OUT / "app_roadanomaly"), "zip", APP_DIR)
print("Contenu du paquet :")
for f in sorted(APP_DIR.iterdir()):
    print(f"  {f.name:<18} {f.stat().st_size / 1e6:7.2f} Mo")

In [ ]:
# Téléchargements : 1) le paquet de l'application (≈ 85 Mo) ; 2) les poids PyTorch (≈ 90 Mo) ;
# 3) les figures et mesures de l'étape 5 (léger)
LEGER = BASE / "resultats_etape5_figures"; LEGER.mkdir(exist_ok=True)
for f in OUT.iterdir():
    if f.suffix in (".png", ".json"):
        shutil.copy(f, LEGER / f.name)
shutil.copy(APP_DIR / "config.json", LEGER / "config.json")
shutil.make_archive(str(BASE / "resultats_etape5_figures"), "zip", LEGER)
if IN_COLAB:
    from google.colab import files
    files.download(str(OUT / "app_roadanomaly.zip"))
    files.download(str(OUT / "model.pt"))
    files.download(str(BASE / "resultats_etape5_figures.zip"))

## Synthèse du déploiement

| Élément | Choix | Justification |
|---|---|---|
| Modèle | DeepLabV3+ / ResNet34, réentraîné sur les 60 images | retenu au benchmark ; plus de données pour le modèle final |
| Époques, seuil | moyennes de la validation croisée | plus de jeu de validation disponible |
| Performance annoncée | celle de la validation croisée | le modèle final n'a plus de jeu de test |
| Format | ONNX + onnxruntime | portable, léger, sans PyTorch, sorties vérifiées identiques |
| Interface | Gradio : image, seuil réglable, zones encadrées, carte de probabilité | simple, utilisable par un non-spécialiste |
| Diffusion | lien public Colab (72 h) et exécution locale (`python app.py`) | gratuit ; démonstration fiable sans internet |

**Pour le rapport (limites et pistes) :**

- **Vitesse :** sur un processeur (CPU), une image prend de l'ordre de la seconde (voir la
  mesure section 3) : très bien pour une démonstration, insuffisant pour un véhicule, qui
  demanderait un GPU embarqué et une version allégée (encodeur plus petit, quantification INT8,
  TensorRT).
- **Domaine :** 60 images, de jour, par beau temps pour la plupart. Les performances sur la nuit,
  la pluie ou d'autres caméras sont inconnues.
- **Suivi :** en production réelle, il faudrait collecter les erreurs signalées par les
  utilisateurs pour enrichir le dataset et réentraîner.